# Data Cleaning - Numeric parts

Notebook này làm sạch các cột tô màu vàng theo phạm vi dữ liệu số. Code giữ nguyên dữ liệu gốc, tạo cột mới và xuất file CSV đã clean.

In [1]:
pip install ipykernel pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Import thư viện cần dùng
import pandas as pd
import numpy as np
import re
from pathlib import Path
import unicodedata


# Hiển thị nhiều cột hơn khi kiểm tra dữ liệu
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

In [3]:
# Khai báo đường dẫn file input/output
INPUT_PATH = Path("../data/chotot_raw.csv")
OUTPUT_CLEANED = Path("chotot_planA_cleaned.csv")
OUTPUT_LOG = Path("planA_parse_log.csv")

# Đọc dữ liệu gốc từ CSV
df = pd.read_csv(INPUT_PATH)

# Xem kích thước dữ liệu
print("Shape:", df.shape)
df.head()

Shape: (9004, 29)


,title,price,area,location,description,Diện tích đất:,Giá/m2:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,Số phòng vệ sinh:,Loại hình nhà ở:,Tình trạng nội thất:,Diện tích sử dụng:,Tình trạng bất động sản:,Diện tích:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"2,38 tỷ- 100 m2",- 100 m2,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ, Đà Nẵng",Còn lô giá rẻ nhất khu vực: Nam Cẩm Lệ\n✔️ Đường Lỗ Giáng 8 - Hoà Xuân \n✔Vị trí song song với đường Mẹ Thứ \n✔Diện ...,100 m2,"23,8 triệu/m2",Nam,Đã có sổ,Mặt tiền,Đất thổ cư,5 m,20 m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm quận 5,18 tỷ- 79 m2,- 79 m2,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","Nhà 1 trệt 2 lầu\nDiện tích 4,15x18,8\n4 phòng ngủ 2 toilet 1 khách\nNội thất cơ bản đầy đủ\nMặt tiền trước nhà rộng...",79 m²,"227,85 triệu/m²",Nam,Đang chờ sổ,Mặt tiền,Đất thổ cư,4 m,18 m,4 phòng,3 phòng,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,89 m²,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2",1 tỷ- 500 m2,- 500 m2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu Bàng, Bình Dương","Vài lô liền kề nằm ngay kcn , tthc bầu bàng \nDiện tích rộng nên có thể đầu tư xây trọ cách các khu công nghiệp chỉ...",500 m2,2 triệu/m2,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,5 m,100 m,4 phòng,3 phòng,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,89 m²,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn",525 triệu- 60 m2,- 60 m2,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Chánh, Tp Hồ Chí Minh","Nhà chính chủ mới xây đường võ văn vân, vĩnh lộc b, bình chánh, diện tích 4x15m, giá bán 525 triệu\ndiện tích 4x15m,...",60 m²,"8,75 triệu/m²",Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,4 m,15 m,3 phòng,2 phòng,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,120 m²,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,440 triệu- 150 m2,- 150 m2,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - Vũng Tàu","bán lô đất đẹp mặt tiền đường nhựa thôn 2, Suối Rao, Châu Đức\nhình chụp thực tế bên trên\nsổ đỏ thực tế bên trên, c...",150 m2,"2,93 triệu/m2",Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6 m,25 m,3 phòng,2 phòng,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,120 m²,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Danh sách cột xử lý - Numberics parts

In [4]:
# Các cột tô màu vàng cần clean trong phần A
plan_a_columns = [
    "price", "area", "Diện tích đất:", "Giá/m2:",
    "Chiều ngang:", "Chiều dài:", "Số phòng ngủ:", "Số phòng vệ sinh:",
    "Diện tích sử dụng:", "Diện tích:", "Tổng số tầng:",
    "Mã căn / Mã căn hộ:", "Tầng số:", "Mã lô:"
]

# Kiểm tra cột nào có trong dataset
existing_plan_a_columns = [c for c in plan_a_columns if c in df.columns]
missing_plan_a_columns = [c for c in plan_a_columns if c not in df.columns]

print("Cột tìm thấy:", existing_plan_a_columns)
print("Cột không tìm thấy:", missing_plan_a_columns)

Cột tìm thấy: ['price', 'area', 'Diện tích đất:', 'Giá/m2:', 'Chiều ngang:', 'Chiều dài:', 'Số phòng ngủ:', 'Số phòng vệ sinh:', 'Diện tích sử dụng:', 'Diện tích:', 'Tổng số tầng:', 'Mã căn / Mã căn hộ:', 'Tầng số:', 'Mã lô:']
Cột không tìm thấy: []


## 2. Hàm chuẩn hóa và parse dữ liệu

In [5]:
# Hàm chuẩn hóa text cơ bản trước khi parse
def normalize_text(value):
    if pd.isna(value):
        return None
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("m²", "m2")
    text = text.replace("㎡", "m2")
    text = text.replace("tỉ", "ty").replace("tỷ", "ty")
    text = re.sub(r"\btriệu\b|\btrieu\b|\btr\b", "trieu", text)
    text = text.replace(",", ".")
    return text

# Hàm parse số đầu tiên trong chuỗi
def first_number(text):
    if text is None:
        return np.nan
    match = re.search(r"\d+(?:\.\d+)?", text)
    if not match:
        return np.nan
    return float(match.group(0))

def parse_price_vnd(value):
    text = normalize_text(value)
    
    if text is None or text == "":
        return np.nan
    
    if any(k in text for k in ["thoa thuan", "thỏa thuận", "thoả thuận", "lien he", "liên hệ"]):
        return np.nan
    
    price_part = text.split("-")[0].strip()


    special = re.search(r"(\d+(?:\.\d+)?)\s*ty\s+(\d+(?:\.\d+)?)", price_part)
    if special:
        billion = float(special.group(1))
        million = float(special.group(2))
        return int(round(billion * 1_000_000_000 + million * 1_000_000))

    number = first_number(price_part)
    if pd.isna(number):
        return np.nan

    # tỷ
    if "ty" in price_part:
        return int(round(number * 1_000_000_000))

    # triệu
    if "trieu" in price_part or "tr" in price_part:
        return int(round(number * 1_000_000))

    if "đ" in price_part:
        # ví dụ: 950.000 đ → 950 triệu
        if number > 100:
            return int(round(number * 1_000_000))

    return np.nan


# Hàm parse diện tích về m2
def parse_area_m2(value):
    text = normalize_text(value)
    if text is None or text == "":
        return np.nan

    text = text.replace("-", " ").strip()

    # bỏ đơn vị
    text = text.replace("m2", " ").replace("m²", " ").replace("m", " ")

    # lấy số đầu tiên
    match = re.search(r"\d+(?:[\.,]\d+)?", text)
    if not match:
        return np.nan

    num_text = match.group(0)

    if re.match(r"^\d+\.\d{3}$", num_text):
        num_text = num_text.replace(".", "")
    else:
        num_text = num_text.replace(",", ".")

    return float(num_text)

# Hàm parse đơn giá về VNĐ/m2
def parse_price_per_m2(value):
    text = normalize_text(value)
    if text is None or text == "":
        return np.nan

    if any(k in text for k in ["thoa thuan", "thỏa thuận", "thoả thuận", "lien he", "liên hệ"]):
        return np.nan

    if "đ" in text or "vnd" in text:
        num_text = re.search(r"\d+(?:[\.,]\d+)*", text)
        if num_text:
            return int(num_text.group(0).replace(".", "").replace(",", ""))

    number = first_number(text)
    if pd.isna(number):
        return np.nan

    if "ty" in text:
        return int(round(number * 1_000_000_000))

    if "trieu" in text or "tr" in text:
        return int(round(number * 1_000_000))

    return np.nan

# Hàm parse chiều ngang/chiều dài về mét
def parse_meter(value):
    text = normalize_text(value)
    if text is None or text == "":
        return np.nan
    return first_number(text)

# Hàm parse số lượng phòng/tầng
def parse_count(value):
    text = normalize_text(value)
    if text is None or text == "":
        return np.nan
    number = first_number(text)
    if pd.isna(number):
        return np.nan
    return int(round(number))

# Hàm chuẩn hóa mã căn/mã lô dạng text, không ép sang số
def clean_code(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    text = re.sub(r"\s+", " ", text)
    return text.upper()

## 3. Tạo các cột clean mới

In [6]:
# Tạo bản copy để không làm thay đổi dữ liệu gốc
clean_df = df.copy()

# Parse giá và diện tích
clean_df["price_vnd"] = clean_df["price"].apply(parse_price_vnd)
clean_df["area_m2"] = clean_df["area"].apply(parse_area_m2)
clean_df["land_area_m2"] = clean_df["Diện tích đất:"].apply(parse_area_m2)
clean_df["price_per_m2_vnd"] = clean_df["Giá/m2:"].apply(parse_price_per_m2)

# Parse kích thước và số phòng/tầng
clean_df["width_m"] = clean_df["Chiều ngang:"].apply(parse_meter)
clean_df["length_m"] = clean_df["Chiều dài:"].apply(parse_meter)
clean_df["bedrooms"] = clean_df["Số phòng ngủ:"].apply(parse_count)
clean_df["toilets"] = clean_df["Số phòng vệ sinh:"].apply(parse_count)
clean_df["usable_area_m2"] = clean_df["Diện tích sử dụng:"].apply(parse_area_m2)
clean_df["apartment_area_m2"] = clean_df["Diện tích:"].apply(parse_area_m2)
clean_df["floors"] = clean_df["Tổng số tầng:"].apply(parse_count)
clean_df["floor_number"] = clean_df["Tầng số:"].apply(parse_count)

# Chuẩn hóa mã căn/mã lô dạng text
clean_df["unit_code_clean"] = clean_df["Mã căn / Mã căn hộ:"].apply(clean_code)
clean_df["lot_code_clean"] = clean_df["Mã lô:"].apply(clean_code)

clean_df[["price", "price_vnd", "area", "area_m2", "Diện tích đất:", "land_area_m2", "Giá/m2:", "price_per_m2_vnd"]].head(10)

,price,price_vnd,area,area_m2,Diện tích đất:,land_area_m2,Giá/m2:,price_per_m2_vnd
0,"2,38 tỷ- 100 m2",2380000000,- 100 m2,100.0,100 m2,100.0,"23,8 triệu/m2",23800000
1,18 tỷ- 79 m2,18000000000,- 79 m2,79.0,79 m²,79.0,"227,85 triệu/m²",227850000
2,1 tỷ- 500 m2,1000000000,- 500 m2,500.0,500 m2,500.0,2 triệu/m2,2000000
3,525 triệu- 60 m2,525000000,- 60 m2,60.0,60 m²,60.0,"8,75 triệu/m²",8750000
4,440 triệu- 150 m2,440000000,- 150 m2,150.0,150 m2,150.0,"2,93 triệu/m2",2930000
5,"1,56 tỷ- 600 m2",1560000000,- 600 m2,600.0,600 m2,600.0,"2,6 triệu/m2",2600000
6,"2,5 tỷ- 67 m2",2500000000,- 67 m2,67.0,600 m2,600.0,"37,31 triệu/m²",37310000
7,890 triệu- 250 m2,890000000,- 250 m2,250.0,250 m2,250.0,"3,56 triệu/m2",3560000
8,560 triệu- 261 m2,560000000,- 261 m2,261.0,261 m2,261.0,"2,15 triệu/m2",2150000
9,25 tỷ- 80 m2,25000000000,- 80 m2,80.0,80 m²,80.0,"312,5 triệu/m²",312500000


## 4. Ghi log lỗi parse và flag inconsistency

In [7]:
# Tạo cột log rỗng
def add_reason(current, reason):
    if pd.isna(current) or current == "":
        return reason
    return str(current) + "; " + reason

clean_df["planA_log"] = ""

# Log lỗi parse giá/diện tích khi cột gốc có dữ liệu nhưng cột clean bị NaN
log_rules = [
    ("price", "price_vnd", "parse_fail_price"),
    ("area", "area_m2", "parse_fail_area"),
    ("Diện tích đất:", "land_area_m2", "parse_fail_land_area"),
    ("Giá/m2:", "price_per_m2_vnd", "parse_fail_price_per_m2"),
    ("Chiều ngang:", "width_m", "parse_fail_width"),
    ("Chiều dài:", "length_m", "parse_fail_length"),
    ("Số phòng ngủ:", "bedrooms", "parse_fail_bedrooms"),
    ("Số phòng vệ sinh:", "toilets", "parse_fail_toilets"),
    ("Diện tích sử dụng:", "usable_area_m2", "parse_fail_usable_area"),
    ("Diện tích:", "apartment_area_m2", "parse_fail_apartment_area"),
    ("Tổng số tầng:", "floors", "parse_fail_floors"),
    ("Tầng số:", "floor_number", "parse_fail_floor_number"),
]

for raw_col, clean_col, reason in log_rules:
    mask = clean_df[raw_col].notna() & clean_df[clean_col].isna()
    clean_df.loc[mask, "planA_log"] = clean_df.loc[mask, "planA_log"].apply(lambda x: add_reason(x, reason))

# Flag diện tích không nhất quán: width * length lệch quá 25% so với diện tích đất hoặc area_m2
calculated_area = clean_df["width_m"] * clean_df["length_m"]
reference_area = clean_df["land_area_m2"].fillna(clean_df["area_m2"])
area_diff_ratio = (calculated_area - reference_area).abs() / reference_area
mask_inconsistent_area = calculated_area.notna() & reference_area.notna() & (reference_area > 0) & (area_diff_ratio > 0.25)
clean_df.loc[mask_inconsistent_area, "planA_log"] = clean_df.loc[mask_inconsistent_area, "planA_log"].apply(lambda x: add_reason(x, "inconsistent_area"))

# Flag đơn giá không nhất quán: price / area lệch quá 35% so với Giá/m2
calculated_ppm = clean_df["price_vnd"] / clean_df["area_m2"]
ppm_diff_ratio = (calculated_ppm - clean_df["price_per_m2_vnd"]).abs() / clean_df["price_per_m2_vnd"]
mask_inconsistent_ppm = calculated_ppm.notna() & clean_df["price_per_m2_vnd"].notna() & (clean_df["price_per_m2_vnd"] > 0) & (ppm_diff_ratio > 0.35)
clean_df.loc[mask_inconsistent_ppm, "planA_log"] = clean_df.loc[mask_inconsistent_ppm, "planA_log"].apply(lambda x: add_reason(x, "inconsistent_price_per_m2"))

# Chuyển log rỗng thành NaN cho dễ lọc
clean_df["planA_log"] = clean_df["planA_log"].replace("", np.nan)

clean_df[clean_df["planA_log"].notna()][["price", "area", "Giá/m2:", "planA_log"]].head(20)

,price,area,Giá/m2:,planA_log
11,"1,4 tỷ- 3.300 m2",- 3.300 m2,424.242 đ/m2,inconsistent_area
18,"1,3 tỷ- 198 m2",- 198 m2,"6,57 triệu/m2",inconsistent_area
21,"2,3 tỷ- 40 m2",- 40 m2,"57,5 triệu/m²",inconsistent_area
25,"4,5 tỷ- 52 m2",- 52 m2,"86,54 triệu/m²",inconsistent_area
30,"7,5 tỷ- 93 m2",- 93 m2,"80,65 triệu/m²",inconsistent_area
36,"2,9 tỷ- 34 m2",- 34 m2,"85,29 triệu/m²",inconsistent_area
42,"4,7 tỷ- 112 m2",- 112 m2,"41,96 triệu/m2",inconsistent_area
43,"2,5 tỷ- 33 m2",- 33 m2,"75,76 triệu/m²",inconsistent_area
49,"4,68 tỷ GIÁ TỐT- 2.600 m2",GIÁ TỐT,"1,8 triệu/m2",parse_fail_area
58,"4,4 tỷ- 240 m2",- 240 m2,"18,33 triệu/m2",inconsistent_area


## 5. Kiểm tra nhanh kết quả clean

In [8]:
# Thống kê số lượng non-null của các cột output
output_columns = [
    "price_vnd", "area_m2", "land_area_m2", "price_per_m2_vnd",
    "width_m", "length_m", "bedrooms", "toilets",
    "usable_area_m2", "apartment_area_m2", "floors", "floor_number",
    "unit_code_clean", "lot_code_clean", "planA_log"
]

summary = pd.DataFrame({
    "non_null": clean_df[output_columns].notna().sum(),
    "null": clean_df[output_columns].isna().sum(),
    "dtype": clean_df[output_columns].dtypes.astype(str)
})
summary

,non_null,null,dtype
price_vnd,9004,0,int64
area_m2,8545,459,float64
land_area_m2,9004,0,float64
price_per_m2_vnd,9004,0,int64
width_m,9004,0,float64
length_m,9004,0,float64
bedrooms,9003,1,float64
toilets,9003,1,float64
usable_area_m2,9003,1,float64
apartment_area_m2,8998,6,float64


In [9]:
# Kiểm tra một vài dòng sau khi clean
preview_cols = [
    "price", "price_vnd", "area", "area_m2", "Diện tích đất:", "land_area_m2",
    "Giá/m2:", "price_per_m2_vnd", "Chiều ngang:", "width_m",
    "Chiều dài:", "length_m", "Số phòng ngủ:", "bedrooms",
    "Số phòng vệ sinh:", "toilets", "Tổng số tầng:", "floors", "planA_log"
]
clean_df[preview_cols].head(20)

,price,price_vnd,area,area_m2,Diện tích đất:,land_area_m2,Giá/m2:,price_per_m2_vnd,Chiều ngang:,width_m,Chiều dài:,length_m,Số phòng ngủ:,bedrooms,Số phòng vệ sinh:,toilets,Tổng số tầng:,floors,planA_log
0,"2,38 tỷ- 100 m2",2380000000,- 100 m2,100.0,100 m2,100.0,"23,8 triệu/m2",23800000,5 m,5.0,20 m,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,18 tỷ- 79 m2,18000000000,- 79 m2,79.0,79 m²,79.0,"227,85 triệu/m²",227850000,4 m,4.0,18 m,18.0,4 phòng,4.0,3 phòng,3.0,NaN,NaN,NaN
2,1 tỷ- 500 m2,1000000000,- 500 m2,500.0,500 m2,500.0,2 triệu/m2,2000000,5 m,5.0,100 m,100.0,4 phòng,4.0,3 phòng,3.0,NaN,NaN,NaN
3,525 triệu- 60 m2,525000000,- 60 m2,60.0,60 m²,60.0,"8,75 triệu/m²",8750000,4 m,4.0,15 m,15.0,3 phòng,3.0,2 phòng,2.0,NaN,NaN,NaN
4,440 triệu- 150 m2,440000000,- 150 m2,150.0,150 m2,150.0,"2,93 triệu/m2",2930000,6 m,6.0,25 m,25.0,3 phòng,3.0,2 phòng,2.0,NaN,NaN,NaN
5,"1,56 tỷ- 600 m2",1560000000,- 600 m2,600.0,600 m2,600.0,"2,6 triệu/m2",2600000,12 m,12.0,50 m,50.0,3 phòng,3.0,2 phòng,2.0,NaN,NaN,NaN
6,"2,5 tỷ- 67 m2",2500000000,- 67 m2,67.0,600 m2,600.0,"37,31 triệu/m²",37310000,12 m,12.0,50 m,50.0,2 phòng,2.0,2 phòng,2.0,NaN,NaN,NaN
7,890 triệu- 250 m2,890000000,- 250 m2,250.0,250 m2,250.0,"3,56 triệu/m2",3560000,10 m,10.0,25 m,25.0,2 phòng,2.0,2 phòng,2.0,NaN,NaN,NaN
8,560 triệu- 261 m2,560000000,- 261 m2,261.0,261 m2,261.0,"2,15 triệu/m2",2150000,15 m,15.0,18 m,18.0,2 phòng,2.0,2 phòng,2.0,NaN,NaN,NaN
9,25 tỷ- 80 m2,25000000000,- 80 m2,80.0,80 m²,80.0,"312,5 triệu/m²",312500000,4 m,4.0,20 m,20.0,6 phòng,6.0,6 phòng,6.0,NaN,NaN,NaN


## Data cleaning - Text parts

## 6. Parse Location column

In [10]:
df[["location"]].head(10)

,location
0,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ, Đà Nẵng"
1,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh"
2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu Bàng, Bình Dương"
3,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Chánh, Tp Hồ Chí Minh"
4,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - Vũng Tàu"
5,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộc, Lâm Đồng"
6,"Đường số 1, Phường Trường Thọ, Quận Thủ Đức, Tp Hồ Chí Minh"
7,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bảo Lâm, Lâm Đồng"
8,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hải Phòng"
9,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Chí Minh"


In [11]:
def clean_location(text):
    if pd.isna(text):
        return None
    
    text = text.lower().strip()
    
    # bỏ ký tự rác
    text = re.sub(r"\|\|\d+", "", text)
    
    # normalize space
    text = re.sub(r"\s+", " ", text)
    
    return text

df["location_clean"] = df["location"].apply(clean_location)

df[["location", "location_clean"]].head(10)

,location,location_clean
0,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ, Đà Nẵng","đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ, đà nẵng"
1,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","đường cao đạt, phường 1, quận 5, tp hồ chí minh"
2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu Bàng, Bình Dương","đường quốc lộ 13, thị trấn lai uyên, huyện bàu bàng, bình dương"
3,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Chánh, Tp Hồ Chí Minh","đường võ văn vân, xã vĩnh lộc b, huyện bình chánh, tp hồ chí minh"
4,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - Vũng Tàu","thôn 2, xã suối rao, huyện châu đức, bà rịa - vũng tàu"
5,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộc, Lâm Đồng","đường lý thái tổ, xã đạm bri, thành phố bảo lộc, lâm đồng"
6,"Đường số 1, Phường Trường Thọ, Quận Thủ Đức, Tp Hồ Chí Minh","đường số 1, phường trường thọ, quận thủ đức, tp hồ chí minh"
7,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bảo Lâm, Lâm Đồng","đường quốc lộ 20, thị trấn lộc thắng, huyện bảo lâm, lâm đồng"
8,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hải Phòng","đường 359, xã tân dương, huyện thuỷ nguyên, hải phòng"
9,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Chí Minh","đường phó cơ điều, phường 12, quận 5, tp hồ chí minh"


In [12]:
def debug_split(text):
    if pd.isna(text):
        return None
    return [p.strip() for p in text.split(",")]

df["parts"] = df["location_clean"].apply(debug_split)

df[["location_clean", "parts"]].head(10)

,location_clean,parts
0,"đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ, đà nẵng","[đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ, đà nẵng]"
1,"đường cao đạt, phường 1, quận 5, tp hồ chí minh","[đường cao đạt, phường 1, quận 5, tp hồ chí minh]"
2,"đường quốc lộ 13, thị trấn lai uyên, huyện bàu bàng, bình dương","[đường quốc lộ 13, thị trấn lai uyên, huyện bàu bàng, bình dương]"
3,"đường võ văn vân, xã vĩnh lộc b, huyện bình chánh, tp hồ chí minh","[đường võ văn vân, xã vĩnh lộc b, huyện bình chánh, tp hồ chí minh]"
4,"thôn 2, xã suối rao, huyện châu đức, bà rịa - vũng tàu","[thôn 2, xã suối rao, huyện châu đức, bà rịa - vũng tàu]"
5,"đường lý thái tổ, xã đạm bri, thành phố bảo lộc, lâm đồng","[đường lý thái tổ, xã đạm bri, thành phố bảo lộc, lâm đồng]"
6,"đường số 1, phường trường thọ, quận thủ đức, tp hồ chí minh","[đường số 1, phường trường thọ, quận thủ đức, tp hồ chí minh]"
7,"đường quốc lộ 20, thị trấn lộc thắng, huyện bảo lâm, lâm đồng","[đường quốc lộ 20, thị trấn lộc thắng, huyện bảo lâm, lâm đồng]"
8,"đường 359, xã tân dương, huyện thuỷ nguyên, hải phòng","[đường 359, xã tân dương, huyện thuỷ nguyên, hải phòng]"
9,"đường phó cơ điều, phường 12, quận 5, tp hồ chí minh","[đường phó cơ điều, phường 12, quận 5, tp hồ chí minh]"


In [14]:
def normalize_text(text):
    text = unicodedata.normalize('NFC', text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [58]:
def starts_with_any(text, keywords):
    return any(text.startswith(k + " ") or text == k for k in keywords)


def parse_location(text):
    if pd.isna(text):
        return pd.Series({
            "street": None,
            "ward": None,
            "district": None,
            "city": None
        })
    
    # normalize
    text = normalize_text(text)
    
    # split
    parts = [p.strip() for p in text.split(",") if p.strip() != ""]
    
    # remove số đầu (vd: "36")
    if len(parts) > 1 and re.fullmatch(r"\d+", parts[0]):
        parts = parts[1:]
        # print("PARTS:", parts)  # debug nếu cần
    
    result = {
        "street": None,
        "ward": None,
        "district": None,
        "city": None
    }
    
    unknown_parts = []
    
    # Keyword detection (prefix-based)
    for part in parts:
        p = part.strip()
        
        if starts_with_any(p, ["phường", "xã", "thị trấn", "thôn", "kênh"]):
            result["ward"] = part
        
        elif starts_with_any(p, ["quận", "huyện", "thị xã"]):
            result["district"] = part
        
        elif starts_with_any(p, ["đường"]):
            result["street"] = part
        
        elif starts_with_any(p, ["tỉnh", "tp"]):
            result["city"] = part
        
        elif p.startswith("thành phố"):
            if result["district"] is None:
                result["district"] = part
            else:
                result["city"] = part
        
        else:
            unknown_parts.append(part)   # FIX INDENT
    
    # Fix city (fallback)
    if result["city"] is None and len(parts) >= 3:
        result["city"] = parts[-1]
    
    # Remove các phần đã dùng
    used = set([v for v in result.values() if v is not None])
    remaining = [p for p in parts if p not in used]
    
    # Fill phần còn lại
    for key in ["street", "ward", "district"]:
        if result[key] is None and remaining:
            result[key] = remaining.pop(0)
    
    return pd.Series(result)

In [59]:
df[["street", "ward", "district", "city"]] = df["location_clean"].apply(parse_location)

df[["location_clean", "street", "ward", "district", "city"]].head(30)

,location_clean,street,ward,district,city
0,"đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ, đà nẵng",đường lỗ giáng 8,phường hòa xuân,quận cẩm lệ,đà nẵng
2,"đường quốc lộ 13, thị trấn lai uyên, huyện bàu bàng, bình dương",đường quốc lộ 13,thị trấn lai uyên,huyện bàu bàng,bình dương
4,"thôn 2, xã suối rao, huyện châu đức, bà rịa - vũng tàu",thôn 2,xã suối rao,huyện châu đức,bà rịa - vũng tàu
5,"đường lý thái tổ, xã đạm bri, thành phố bảo lộc, lâm đồng",đường lý thái tổ,xã đạm bri,thành phố bảo lộc,lâm đồng
6,"đường số 1, phường trường thọ, quận thủ đức, tp hồ chí minh",đường số 1,phường trường thọ,quận thủ đức,tp hồ chí minh
7,"đường quốc lộ 20, thị trấn lộc thắng, huyện bảo lâm, lâm đồng",đường quốc lộ 20,thị trấn lộc thắng,huyện bảo lâm,lâm đồng
8,"đường 359, xã tân dương, huyện thuỷ nguyên, hải phòng",đường 359,xã tân dương,huyện thuỷ nguyên,hải phòng
9,"đường phó cơ điều, phường 12, quận 5, tp hồ chí minh",đường phó cơ điều,phường 12,quận 5,tp hồ chí minh
10,"đường số 8, phường hiệp bình phước, quận thủ đức, tp hồ chí minh",đường số 8,phường hiệp bình phước,quận thủ đức,tp hồ chí minh
11,"đường lý thường kiệt, phường lộc phát, thành phố bảo lộc, lâm đồng",đường lý thường kiệt,phường lộc phát,thành phố bảo lộc,lâm đồng


In [60]:
test = "Thôn Lộc Châu 2, Xã Tân Nghĩa, Huyện Di Linh, Lâm Đồng"

print(parse_location(clean_location(test)))

street      thôn lộc châu 2
ward           xã tân nghĩa
district      huyện di linh
city               lâm đồng
dtype: str


In [61]:
df[df["city"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,NaN,NaN,NaN
50,Đường Phạm Văn Đồng,đường phạm văn đồng,NaN,NaN,NaN
116,Đường Đinh Đức Thiện,đường đinh đức thiện,NaN,NaN,NaN
165,Đường D7,đường d7,NaN,NaN,NaN
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,NaN,NaN,NaN
261,Đường Bùi Đình Túy,đường bùi đình túy,NaN,NaN,NaN
368,Hương An,hương an,NaN,NaN,NaN
414,D6,d6,NaN,NaN,NaN
679,Đường D7,đường d7,NaN,NaN,NaN
785,Duong go xoai,duong go xoai,NaN,NaN,NaN


In [62]:
df[df["district"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,NaN,NaN,NaN
50,Đường Phạm Văn Đồng,đường phạm văn đồng,NaN,NaN,NaN
116,Đường Đinh Đức Thiện,đường đinh đức thiện,NaN,NaN,NaN
165,Đường D7,đường d7,NaN,NaN,NaN
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,NaN,NaN,NaN
261,Đường Bùi Đình Túy,đường bùi đình túy,NaN,NaN,NaN
368,Hương An,hương an,NaN,NaN,NaN
414,D6,d6,NaN,NaN,NaN
679,Đường D7,đường d7,NaN,NaN,NaN
785,Duong go xoai,duong go xoai,NaN,NaN,NaN


In [63]:
df[df["ward"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,NaN,NaN,NaN
50,Đường Phạm Văn Đồng,đường phạm văn đồng,NaN,NaN,NaN
116,Đường Đinh Đức Thiện,đường đinh đức thiện,NaN,NaN,NaN
165,Đường D7,đường d7,NaN,NaN,NaN
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,NaN,NaN,NaN
261,Đường Bùi Đình Túy,đường bùi đình túy,NaN,NaN,NaN
368,Hương An,hương an,NaN,NaN,NaN
414,D6,d6,NaN,NaN,NaN
679,Đường D7,đường d7,NaN,NaN,NaN
785,Duong go xoai,duong go xoai,NaN,NaN,NaN


In [64]:
df[df["street"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
834,"200, Xã Bình Hiệp, Huyện Bình Sơn, Quảng Ngãi",NaN,xã bình hiệp,huyện bình sơn,quảng ngãi
1394,"301, Xã Tân Thạnh Đông, Huyện Củ Chi, Tp Hồ Chí Minh",NaN,xã tân thạnh đông,huyện củ chi,tp hồ chí minh
1693,"322, Xã Tân Phước, Huyện Đồng Phú, Bình Phước",NaN,xã tân phước,huyện đồng phú,bình phước
1719,"322, Xã Tân Phước, Huyện Đồng Phú, Bình Phước",NaN,xã tân phước,huyện đồng phú,bình phước
2120,"xã Duy Nghĩa, Xã Duy Nghĩa, Huyện Duy Xuyên, Quảng Nam",NaN,xã duy nghĩa,huyện duy xuyên,quảng nam
2327,"980, Phường Phú Hữu, Quận 9, Tp Hồ Chí Minh",NaN,phường phú hữu,quận 9,tp hồ chí minh
2526,"1, Xã An Linh, Huyện Phú Giáo, Bình Dương",NaN,xã an linh,huyện phú giáo,bình dương
2532,"943, Xã Vĩnh Thành, Huyện Châu Thành, An Giang",NaN,xã vĩnh thành,huyện châu thành,an giang
2791,"713, Thị trấn Đạ M'ri, Huyện Đạ Huoai, Lâm Đồng",NaN,thị trấn đạ m'ri,huyện đạ huoai,lâm đồng
3331,"105, Phường Tân Phú, Quận 9, Tp Hồ Chí Minh",NaN,phường tân phú,quận 9,tp hồ chí minh


In [65]:
parse_location("Đường Tỉnh lộ 15")

street      đường tỉnh lộ 15
ward                     NaN
district                 NaN
city                     NaN
dtype: str

## 7. Parse Tên phân khu/ lô/ Block, Tháp Column

In [79]:
# Clear Data
def clean_text(x):
    if pd.isna(x):
        return x
    x = x.upper().strip()
    x = re.sub(r"\s+", " ", x)
    return x

df["ten_phan_khu_clean"] = df["Tên phân khu/Lô/Block/Tháp:"].apply(clean_text)


In [80]:
df["ten_phan_khu_clean"].value_counts()

ten_phan_khu_clean
A                                1066
B                                 728
THÔN BÌNH KHÁNH                   275
1                                 261
C                                 222
                                 ... 
L                                   1
THỬA 820                            1
CAM LÂM, NHA TRANG, KHÁNH HÒA       1
103 NGUYỄN THỊ THÂPH                1
LÔ B                                1
Name: count, Length: 242, dtype: int64

In [39]:
for val in df["ten_phan_khu_clean"].dropna().unique():
    print(val)

DỰ ÁN TÍN HƯNG
B
KHU DÂN CƯ AN PHÚ HƯNG
BỆNH VIỆN XUYÊN Á TÂY NINH
A
C
PHƯỚC BÌNH
ABCD
AN BÌNH
BAU BÀNG
KHU ĐƯỜNG BÀN CỜ PHÚ THẠNH-PHÚ THỌ HÒA
L43, L44, L45
RIGEL
D1
115
ĐẤT TÂN HIỆP
KBD
THÔN BÌNH KHÁNH
KDC
DỊCH VỤ 6.9
D
SÀI GÒN VILLAGE
BABYLON
NHÀ DOI DIEN CONG VIEN,RAT MAT ME.
MP1
B3
KHU ĐÔ THỊ TRƯỜNG AN
A,B
DCH
NHÀ PHỐ LIỀN KỀ
E3
KHU HOÀNG HOA THÁM
A B
KHU DÂN CU
1
HOMELAND PARADISE VILLA
2
TOM 77
H
KDCD
NAM CẨM LỆ
MANHATTAN
C4,C5,A4
N16
H20
HẺM 481 ĐƯỜNG TÂN KỲ TÂN QUÝ
HANEL SÀI ĐỒNG LONG BIÊN
3
749
E
NGỌC ĐỊNH FARM
T5
LÔ E
LIỀN KHU DÂN CƯ THỊNH VƯỢNG
CHÂU THỚI
ĐẠI THÀNH NGHI KIM
NGÕ 37 ĐẠI ĐỒNG
DỰ ÁN KHU NGHĨ DƯỠNG BÃI DÀI PHÚ QUỐC
0
A8
BLOCK B
VĨNH ĐIỀM THƯỢNG
ẤP 1 SÔNG TRẦU THỊ TRẤN TRẢNG BOM
KHU DÂN CƯ
KHU DÂN CƯ THẠNH MỸ LỢI DRAGON
BÌNH KHÁNH 2
S2.02
ĐẢO THỊNH VƯỢNG TAM ĐA
ĐƯỜNG 4A
TA15
THE MANHATTAN GLORY
BÌNH NGUYÊN
A1
KHU DAN CƯ CAO CÂP
6-MAY
MẶT TIỀN 833
G50
HẺM TRỊNH ĐÌNH TRỌNG
L K J H F G
CHÙA ĐỨC VIÊN
B2.11
965
NHÀ HẺM
NAM LONG
KHU DÂN CƯ NINH GIANG CÁT LÁI
CT1
MẶT PHỐ


In [81]:
# Classify Data
def classify_type(x):
    if pd.isna(x):
        return "other"
    
    if any(k in x for k in [
        "KHU", "DỰ ÁN", "VILLAGE", "CITY", "RESIDENCE", "RESIDENCES",
        "PARADISE", "RIVERSIDE", "URBAN", "KDC"
    ]):
        return "project"
    
    if re.fullmatch(r"[A-Z]\d{0,2}", x):
        return "block"
    
    if re.fullmatch(r"(LÔ|BLOCK)\s*[A-Z0-9]+", x):
        return "block"
    
    if re.fullmatch(r"[A-Z](\s+[A-Z0-9]+)+", x):
        return "block"
    
    return "other"

In [82]:
df["phan_loai"] = df["ten_phan_khu_clean"].apply(classify_type)

In [83]:
# NORMALIZE BLOCK
def normalize_block(x):
    if pd.isna(x):
        return x
    parts = re.split(r"\s+", x)
    parts = sorted(set(parts))
    return ",".join(parts)

In [91]:
def normalize_block(x):
    parts = re.split(r"[,\s]+", x)
    parts = sorted(set(parts))
    return ",".join(parts)

df.loc[df["phan_loai"] == "block", "ten_phan_khu_final"] = (
    df.loc[df["phan_loai"] == "block", "ten_phan_khu_clean"]
    .apply(normalize_block)
)

In [92]:
df[["Tên phân khu/Lô/Block/Tháp:","ten_phan_khu_clean", "phan_loai", "ten_phan_khu_final"]].head(40)

,Tên phân khu/Lô/Block/Tháp:,ten_phan_khu_clean,phan_loai,ten_phan_khu_final
0,NaN,NaN,other,Không thuộc project/block
2,NaN,NaN,other,Không thuộc project/block
4,NaN,NaN,other,Không thuộc project/block
5,NaN,NaN,other,Không thuộc project/block
6,NaN,NaN,other,Không thuộc project/block
7,NaN,NaN,other,Không thuộc project/block
8,NaN,NaN,other,Không thuộc project/block
9,NaN,NaN,other,Không thuộc project/block
10,NaN,NaN,other,Không thuộc project/block
11,NaN,NaN,other,Không thuộc project/block


In [86]:
# FINAL COLUMN
df["ten_phan_khu_final"] = None

# project giữ nguyên
df.loc[df["phan_loai"] == "project", "ten_phan_khu_final"] = df["ten_phan_khu_clean"]

# block normalize
df.loc[df["phan_loai"] == "block", "ten_phan_khu_final"] = (
    df.loc[df["phan_loai"] == "block", "ten_phan_khu_clean"]
    .apply(normalize_block)
)

# other
df.loc[df["phan_loai"] == "other", "ten_phan_khu_final"] = "Không thuộc project/block"

## 8. Xuất file bàn giao

In [ ]:
# 1. Drop rows "Nhà" không có Loại hình căn hộ (Part B transform)
df = df[
    ~(
        df["Loại hình căn hộ:"].isna() &
        df["title"].str.contains("Nhà", case=False, na=False)
    )
]

# 2. Fillna "other" cho các cột category thưa (Part B transform)
for col in ["Hướng ban công:", "Đặc điểm căn hộ:", "Loại hình văn phòng:"]:
    df[col] = df[col].fillna("other").astype(str).str.strip().replace("", "other")

# 3. Bỏ các row đã drop ở df ra khỏi clean_df (align index)
clean_df = clean_df[clean_df.index.isin(df.index)]

# 4. Sync các cột text clean từ df → clean_df
text_cols_from_df = [
    "location_clean", "street", "ward", "district", "city",
    "ten_phan_khu_clean", "phan_loai", "ten_phan_khu_final",
    "Hướng ban công:", "Đặc điểm căn hộ:", "Loại hình văn phòng:"
]
for col in text_cols_from_df:
    if col in df.columns:
        clean_df[col] = df[col]   # không dùng .values

# 5. Ghi đè cột numeric clean vào cột gốc
clean_to_original = {
    "price_vnd":          "price",
    "area_m2":            "area",
    "land_area_m2":       "Diện tích đất:",
    "price_per_m2_vnd":   "Giá/m2:",
    "width_m":            "Chiều ngang:",
    "length_m":           "Chiều dài:",
    "bedrooms":           "Số phòng ngủ:",
    "toilets":            "Số phòng vệ sinh:",
    "floors":             "Tổng số tầng:",
    "usable_area_m2":     "Diện tích sử dụng:",
    "apartment_area_m2":  "Diện tích:",
    "floor_number":       "Tầng số:",
    "unit_code_clean":    "Mã căn / Mã căn hộ:",
    "lot_code_clean":     "Mã lô:",
}
for clean_col, original_col in clean_to_original.items():
    if clean_col in clean_df.columns:
        clean_df[original_col] = clean_df[clean_col]

# 6. Chọn cột và export
raw_columns = pd.read_csv(INPUT_PATH, nrows=0).columns.tolist()
extra_cols = ["location_clean", "street", "ward", "district", "city",
              "ten_phan_khu_final", "planA_log"]
final_cols = raw_columns + [c for c in extra_cols if c in clean_df.columns]
df_final = clean_df[[col for col in final_cols if col in clean_df.columns]]

OUTPUT_FINAL = Path("../data/chotot_clean.csv")
df_final.to_csv(OUTPUT_FINAL, index=False, encoding="utf-8-sig")
print(f" Exported: {OUTPUT_FINAL}")
print(f"   Rows: {len(df_final):,}  |  Cols: {len(df_final.columns)}")
df_final.head(3)

 Exported: chotot_clean.csv
   Rows: 9,002  |  Cols: 36


,title,price,area,location,description,Diện tích đất:,Giá/m2:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,Số phòng vệ sinh:,Loại hình nhà ở:,Tình trạng nội thất:,Diện tích sử dụng:,Tình trạng bất động sản:,Diện tích:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:,location_clean,street,ward,district,city,ten_phan_khu_final,planA_log
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,2380000000,100.0,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ, Đà Nẵng",Còn lô giá rẻ nhất khu vực: Nam Cẩm Lệ\n✔️ Đường Lỗ Giáng 8 - Hoà Xuân \n✔Vị trí song song với đường Mẹ Thứ \n✔Diện ...,100.0,23800000,Nam,Đã có sổ,Mặt tiền,Đất thổ cư,5.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other,NaN,other,other,"đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ, đà nẵng",đường lỗ giáng 8,phường hòa xuân,quận cẩm lệ,đà nẵng,Không thuộc project/block,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2",1000000000,500.0,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu Bàng, Bình Dương","Vài lô liền kề nằm ngay kcn , tthc bầu bàng \nDiện tích rộng nên có thể đầu tư xây trọ cách các khu công nghiệp chỉ...",500.0,2000000,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,5.0,100.0,4.0,3.0,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,89.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other,NaN,other,other,"đường quốc lộ 13, thị trấn lai uyên, huyện bàu bàng, bình dương",đường quốc lộ 13,thị trấn lai uyên,huyện bàu bàng,bình dương,Không thuộc project/block,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,440000000,150.0,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - Vũng Tàu","bán lô đất đẹp mặt tiền đường nhựa thôn 2, Suối Rao, Châu Đức\nhình chụp thực tế bên trên\nsổ đỏ thực tế bên trên, c...",150.0,2930000,Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,6.0,25.0,3.0,2.0,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other,NaN,other,other,"thôn 2, xã suối rao, huyện châu đức, bà rịa - vũng tàu",thôn 2,xã suối rao,huyện châu đức,bà rịa - vũng tàu,Không thuộc project/block,NaN
